# 03 — Flight Price EDA and Feature Engineering Notes

### Turn travel details into useful, model-friendly features

**Level:** beginner → advanced  
**Dataset:** the supplied `flight_price.xlsx` workbook.

> **Simple picture:** a flight row is a travel ticket. The model cannot understand a clock string like `22:20` or a sentence like `2 stops`, so we gently turn them into useful number clues.

## Learning goals

You will inspect the Flight Price dataset, parse dates and times, create duration and stop features, handle categories, and prepare a safe preprocessing pipeline.


## 1. What is in the flight dataset?

The supplied workbook contains flight details such as airline, journey date, source, destination, route, departure time, arrival time, duration, number of stops, and price.

| Original column | What it means | A useful new clue |
| --- | --- | --- |
| `Date_of_Journey` | Travel date written as text | Journey day, month, weekday. |
| `Dep_Time` / `Arrival_Time` | Clock time as text | Departure and arrival hour/minute. |
| `Duration` | Text such as `2h 50m` | Total journey minutes. |
| `Total_Stops` | Text such as `1 stop` | A number from 0 upward. |
| Airline/source/destination | Category labels | One-hot encoded columns. |

The target is normally `Price` when you want to predict how expensive a flight may be.


In [ ]:
# Beginner-friendly guide:
# These libraries read the Excel file, clean text, and prepare columns for a machine-learning model.
# Path makes the notebook work with the copied workbook in the same folder.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

plt.style.use("seaborn-v0_8-whitegrid")


In [ ]:
# Beginner-friendly guide:
# We read the supplied Excel workbook and make a copy so the original table stays unchanged.
# The first checks show column names, data types, missing values, and a few example tickets.
file_path = Path("flight_price.xlsx")
if not file_path.exists():
    raise FileNotFoundError("flight_price.xlsx should be in the same folder as this notebook.")

flight = pd.read_excel(file_path).copy()
print("Shape:", flight.shape)
display(flight.head())
display(flight.isna().sum().rename("missing_count"))


## 2. Dates and times are hidden treasure

Dates and clocks look like text, but they contain useful patterns.

- The **month** can capture seasonality.
- The **day of week** can capture weekday versus weekend travel.
- Departure hour can distinguish early-morning, daytime, and late-night flights.
- Duration tells us how long the trip lasts in one clean number.

Always parse dates with a real date function instead of cutting characters by position. Real parsing is safer when formats change.


In [ ]:
# Beginner-friendly guide:
# We turn the journey-date text into a real date, then gently take out day, month, and weekday numbers.
# dayofweek uses Monday=0 through Sunday=6, which a model can use as a small category-like number.
flight["Date_of_Journey"] = pd.to_datetime(flight["Date_of_Journey"], dayfirst=True, errors="coerce")
flight["journey_day"] = flight["Date_of_Journey"].dt.day
flight["journey_month"] = flight["Date_of_Journey"].dt.month
flight["journey_weekday"] = flight["Date_of_Journey"].dt.dayofweek

display(flight[["Date_of_Journey", "journey_day", "journey_month", "journey_weekday"]].head())


In [ ]:
# Beginner-friendly guide:
# Arrival time sometimes includes an extra date, so we keep only the clock part before converting it.
# A real time parser gives us separate hour and minute features without guessing from string positions.
arrival_clock = flight["Arrival_Time"].astype("string").str.split().str[0]
departure_clock = flight["Dep_Time"].astype("string").str.split().str[0]

arrival_time = pd.to_datetime(arrival_clock, format="%H:%M", errors="coerce")
departure_time = pd.to_datetime(departure_clock, format="%H:%M", errors="coerce")

flight["arrival_hour"] = arrival_time.dt.hour
flight["arrival_minute"] = arrival_time.dt.minute
flight["departure_hour"] = departure_time.dt.hour
flight["departure_minute"] = departure_time.dt.minute

display(flight[["Dep_Time", "Departure_hour"]] if "Departure_hour" in flight else flight[["Dep_Time", "departure_hour", "arrival_hour"]].head())


In [ ]:
# Beginner-friendly guide:
# Duration text such as "2h 50m" is split into hours and minutes, then changed into one total-minute number.
# Using one unit makes a short flight and a long flight easy for a model to compare.
duration_parts = flight["Duration"].astype("string").str.extract(r"(?:(?P<hours>\d+)h)?\s*(?:(?P<minutes>\d+)m)?")
flight["duration_minutes"] = (
    pd.to_numeric(duration_parts["hours"], errors="coerce").fillna(0) * 60
    + pd.to_numeric(duration_parts["minutes"], errors="coerce").fillna(0)
)

display(flight[["Duration", "duration_minutes"]].head())


## 3. Stops and categories

`Total_Stops` is written as words, but it has a true order: non-stop is 0, then 1 stop, 2 stops, and so on. That makes an ordered numeric mapping sensible.

Airline, source, and destination do **not** have a natural number order. One-hot encoding is safer for these labels because it gives each category its own yes/no column.

### Important decision

`Route` can contain similar information to source, destination, and stops. Keep it only if it adds information that your final model needs; otherwise it may be redundant or too detailed.


In [ ]:
# Beginner-friendly guide:
# We map stop labels to their natural count. Missing stops stay missing so we can decide later how to handle them.
# We also make a quick price plot to see whether flights with more stops tend to have different prices.
stop_map = {"non-stop": 0, "1 stop": 1, "2 stops": 2, "3 stops": 3, "4 stops": 4}
flight["total_stops_number"] = flight["Total_Stops"].map(stop_map)

display(flight[["Total_Stops", "total_stops_number"]].head())
sns.boxplot(data=flight, x="total_stops_number", y="Price")
plt.title("Price distribution by number of stops")
plt.show()


In [ ]:
# Beginner-friendly guide:
# We build a reusable recipe: fill missing values, keep numeric columns, and one-hot encode word categories.
# fit_transform learns category lists from this data; in a real model, fit it on training data only.
model_columns = [
    "journey_day", "journey_month", "journey_weekday", "departure_hour",
    "arrival_hour", "duration_minutes", "total_stops_number", "Airline", "Source", "Destination"
]
X = flight[model_columns]
y = flight["Price"]

numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("numbers", SimpleImputer(strategy="median"), numeric_features),
    ("categories", Pipeline([
        ("fill", SimpleImputer(strategy="most_frequent")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## 4. Advanced safety notes

- Split train and test data **before** fitting imputers, encoders, scalers, or feature-selection rules.
- Use `handle_unknown="ignore"` so a new airline or city does not crash a saved model.
- Check for extreme prices. A rare luxury ticket can be real, so investigate before removing it.
- Keep a clear list of the columns used by the model. That list becomes part of your model’s contract.

## End-of-topic recap

Feature engineering changes travel details into consistent clues: dates become calendar parts, clocks become hours/minutes, duration becomes minutes, stops become counts, and categories become safe indicator columns.
